# W09 — Assignment único semanal (Limpieza avanzada + Quality Gates)

## Setup

In [ ]:
from pathlib import Path
import duckdb

PROJECT_ROOT = Path("/home/manuela-rios/Documentos/Física computacional/Manuela-Rios")
DB_PATH = PROJECT_ROOT / "data" / "exoplanets.duckdb"
RAW_CSV = PROJECT_ROOT / "data" / "raw" / "pscomppars.csv"

if not DB_PATH.exists():
    raise FileNotFoundError(f"Missing {DB_PATH}. Run W06 pipeline first.")

con = duckdb.connect(str(DB_PATH))

def sql_path(p: Path) -> str:
    return "'" + p.resolve().as_posix().replace("'","''") + "'"

if not RAW_CSV.exists():
    raise FileNotFoundError(f"Missing {RAW_CSV}")

con.execute("DROP VIEW IF EXISTS raw_ps")
con.execute(f"CREATE VIEW raw_ps AS SELECT * FROM read_csv_auto({sql_path(RAW_CSV)})")


## Parte A — Limpieza avanzada

In [ ]:
# TODO 1: method_synonyms(raw_norm, canonical)
# - raw_norm: normalizado (LOWER/TRIM + colapsar espacios si quieres)
# - canonical: snake_case
# - >= 6 filas

con.execute("DROP TABLE IF EXISTS method_synonyms")
con.execute("CREATE TABLE method_synonyms(raw_norm VARCHAR, canonical VARCHAR)")
con.execute("""INSERT INTO method_synonyms VALUES
  ('transit', 'transit'),
  ('radial velocity', 'radial_velocity'),
  ('imaging', 'imaging'),
  ('microlensing', 'microlensing'),
  ('timing', 'timing'),
  ('transit timing variations', 'transit_timing_variations'),
  ('eclipse timing variations', 'eclipse_timing_variations'),
  ('orbital brightness modulation', 'orbital_brightness_modulation')
""")

con.sql("SELECT * FROM method_synonyms").show()


In [ ]:
# TODO 2: silver_planet_v3
# Debe incluir:
# - hostname_canon
# - discoverymethod_canon (synonyms + COALESCE fallback)
# - disc_year_int = TRY_CAST(disc_year AS INTEGER)
# - disc_year_bad flag

con.execute("DROP TABLE IF EXISTS silver_planet_v3")
con.execute("""CREATE TABLE silver_planet_v3 AS
WITH base AS (
  SELECT
    pl_name,
    hostname,
    discoverymethod,
    disc_year,
    sy_snum,
    sy_pnum,
    sy_dist,
    ra,
    dec,
    pl_orbper,
    pl_rade,
    pl_bmasse,
    pl_eqt,
    st_teff,
    st_rad,
    st_mass,
    LOWER(TRIM(hostname)) AS hostname_canon,
    LOWER(REGEXP_REPLACE(TRIM(COALESCE(discoverymethod, '')), '\s+', ' ')) AS discoverymethod_norm,
    TRY_CAST(disc_year AS INTEGER) AS disc_year_int
  FROM raw_ps
  WHERE pl_name IS NOT NULL
    AND hostname IS NOT NULL
),
mapped AS (
  SELECT
    b.*,
    COALESCE(ms.canonical, b.discoverymethod_norm) AS discoverymethod_canon,
    CASE
      WHEN b.disc_year IS NULL THEN FALSE
      WHEN TRY_CAST(b.disc_year AS INTEGER) IS NULL THEN TRUE
      WHEN TRY_CAST(b.disc_year AS INTEGER) < 1980 OR TRY_CAST(b.disc_year AS INTEGER) > 2026 THEN TRUE
      ELSE FALSE
    END AS disc_year_bad
  FROM base b
  LEFT JOIN method_synonyms ms
    ON ms.raw_norm = b.discoverymethod_norm
)
SELECT * FROM mapped
""")

con.sql("SELECT COUNT(*) AS n_rows FROM silver_planet_v3").show()
con.sql("SELECT COUNT(*) AS disc_year_bad FROM silver_planet_v3 WHERE disc_year_bad").show()


## Parte B — Quality gates

In [ ]:
# TODO 3: quality_events + 4 checks
# Crea tabla:
# ts_utc, check_name, status, metric_value, details

con.execute("DROP TABLE IF EXISTS quality_events")
con.execute("""
CREATE TABLE quality_events (
  ts_utc TIMESTAMP,
  check_name VARCHAR,
  status VARCHAR,
  metric_value BIGINT,
  details VARCHAR
)
""")

con.execute("""
INSERT INTO quality_events
SELECT CURRENT_TIMESTAMP, 'null_hostname_canon',
       CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END,
       COUNT(*),
       'Rows with hostname_canon IS NULL'
FROM silver_planet_v3
WHERE hostname_canon IS NULL
UNION ALL
SELECT CURRENT_TIMESTAMP, 'disc_year_bad_rows',
       CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'WARN' END,
       COUNT(*),
       'Rows flagged as disc_year_bad'
FROM silver_planet_v3
WHERE disc_year_bad
UNION ALL
SELECT CURRENT_TIMESTAMP, 'null_discoverymethod_canon',
       CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'WARN' END,
       COUNT(*),
       'Rows with empty or null canonical method'
FROM silver_planet_v3
WHERE discoverymethod_canon IS NULL OR discoverymethod_canon = ''
UNION ALL
SELECT CURRENT_TIMESTAMP, 'invalid_pl_rade_range',
       CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'WARN' END,
       COUNT(*),
       'Rows with pl_rade <= 0 or > 30'
FROM silver_planet_v3
WHERE pl_rade IS NOT NULL AND (pl_rade <= 0 OR pl_rade > 30)
""")

con.sql("SELECT check_name, status, metric_value FROM quality_events ORDER BY check_name").show()


## Entregable único semanal (W09)

Entrega:
- `assignments/W09_assignment_student.ipynb` ejecutado
- `docs/w09_report.md` (usar template)
- `docs/w09_quality.md` (usar template)
- 1 entrada en `docs/decisions_log.md` (usar template)